In [0]:
from pyspark.sql.functions import *

#we're reading the source delta tables here
trip_df = spark.table("demodb.trip_data")
zone_df = spark.table("demodb.taxi_zones")

#next, we're adding pickup zone information
pickup_zone_df = (
    trip_df
    .join(
        zone_df,
        trip_df.PULocationID == zone_df.LocationID,
        "left"
    )
    .select(
        trip_df["*"],
        col("Borough").alias("pickup_borough"),
        col("Zone").alias("pickup_zone"),
        col("service_zone").alias("pickup_service_zone")
    )
)

#next, we're adding drop off zone information
dropoff_zone_df = (
    zone_df
    .select(
        col("LocationID").alias("DOLocationID"),
        col("Borough").alias("dropoff_borough"),
        col("Zone").alias("dropoff_zone"),
        col("service_zone").alias("dropoff_service_zone")
    )
)

#next, we create a master dataframe by joining the two dataframes
master_df = (
    pickup_zone_df
    .join(
        dropoff_zone_df,
        "DOLocationID",
        "left"
    )
)

#we calculate the trip duration in minutes (rounded off to the nearest decimal point)
master_df = master_df.withColumn(
    "trip_duration_minutes",
    round(
        (
            unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime")
        ) / 60,
        1
    )
)

#similarly, we calculate average speed (rounded off to the nearest decimal point)
master_df = master_df.withColumn(
    "average_speed_mph",
    round(
        when(
            col("trip_duration_minutes") > 0,
            col("trip_distance") /
            (col("trip_duration_minutes") / 60)
        ).otherwise(None),
        1
    )
)

#we're also calculating fare per mile (rounded off to the nearest decimal point)
master_df = master_df.withColumn(
    "fare_per_mile",
    round(
        when(
            col("trip_distance") > 0,
            col("total_amount") / col("trip_distance")
        ).otherwise(None),
        1
    )
)

#we categorize the trip distance as invalid, short, medium, long or very long
master_df = master_df.withColumn(
    "trip_distance_category",
    when(
        col("trip_distance") <= 0,
        "Invalid"
    )
    .when(
        col("trip_distance") <= 1,
        "Short"
    )
    .when(
        col("trip_distance") <= 5,
        "Medium"
    )
    .when(
        col("trip_distance") <= 10,
        "Long"
    )
    .otherwise("Very Long")
)

#we add a pickup date column
master_df = master_df.withColumn(
    "pickup_date",
    to_date(col("tpep_pickup_datetime"))
)

#we add a pickup hour column
master_df = master_df.withColumn(
    "pickup_hour",
    hour(col("tpep_pickup_datetime"))
)

#now we add a data quality reason column
master_df = master_df.withColumn(
    "data_quality_reason",
    concat_ws(
        ", ",

        #missing pickup or drop-off timestamp
        when(
            col("tpep_pickup_datetime").isNull() |
            col("tpep_dropoff_datetime").isNull(),
            "Missing timestamp"
        ),

        #suspicious historical dates
        when(
            year(col("tpep_pickup_datetime")) < 2023,
            "Suspicious pickup date"
        ),

        #zero or negative trip distance
        when(
            col("trip_distance") <= 0,
            "Invalid trip distance"
        ),

        #zero or negative trip duration
        when(
            col("trip_duration_minutes") <= 0,
            "Invalid trip duration"
        ),

        #trips longer than 3 hours
        when(
            col("trip_duration_minutes") > 180,
            "Excessive trip duration"
        ),

        #negative transaction amount
        when(
            col("total_amount") < 0,
            "Negative total amount"
        ),

        #zero or negative passenger count
        when(
            col("passenger_count") <= 0,
            "Invalid passenger count"
        ),

        #unrealistic average speed
        when(
            col("average_speed_mph") > 100,
            "Unrealistic speed"
        ),

        #unrealistic fare per mile
        when(
            col("fare_per_mile") > 100,
            "Unrealistic fare per mile"
        )
    )
)

#now, we add a data quality flag that either says valid or invalid based on the data quality reason column
master_df = master_df.withColumn(
    "data_quality_flag",
    when(
        length(col("data_quality_reason")) > 0,
        "Invalid"
    )
    .otherwise("Valid")
)

#we view the final master dataframe
display(master_df)

master_df.printSchema()

#we create a database, remove the old master table and save the final master dataframe as a table and verify it
spark.sql(
    "CREATE DATABASE IF NOT EXISTS demodb"
)

spark.sql(
    "DROP TABLE IF EXISTS demodb.nyc_taxi_master"
)

master_df.write \
    .mode("overwrite") \
    .saveAsTable("demodb.nyc_taxi_master")

saved_master_df = spark.table(
    "demodb.nyc_taxi_master"
)

saved_master_df.printSchema()

display(
    saved_master_df.limit(20)
)

#data quality flag summary
spark.sql("""
    SELECT
        data_quality_flag,
        COUNT(*) AS record_count
    FROM demodb.nyc_taxi_master
    GROUP BY data_quality_flag
    ORDER BY data_quality_flag
""").show()

#data quality reason summary
spark.sql("""
    SELECT
        data_quality_reason,
        COUNT(*) AS record_count
    FROM demodb.nyc_taxi_master
    WHERE data_quality_flag = 'Invalid'
    GROUP BY data_quality_reason
    ORDER BY record_count DESC
""").show()

#finally, we check for extreme values
spark.sql("""
    SELECT
        MAX(trip_duration_minutes) AS max_duration_minutes,
        MAX(average_speed_mph) AS max_speed_mph,
        MAX(fare_per_mile) AS max_fare_per_mile
    FROM demodb.nyc_taxi_master
""").show()